# CSE5280 — Evacuation Simulation (Three-Floor Building)

**Author:** Joshua Cajuste
**Course:** CSE5280
**Instructor:** Eraldo Ribeiro
**Department of Electrical Engineering and Computer Science, Florida Institute of Technology**

---

## Overview

This notebook implements a multi-particle evacuation simulation of a three-floor building using a cost-function minimization framework. All agent motion comes purely from gradient descent on a scalar cost function — no collision detection, no path-planning algorithms, and no hard-coded floor switching. Every behavior is emergent from the cost landscape.

The simulation includes:
- 20 agents randomly distributed across three floors
- Two exits on the ground floor with soft-min selection so agents split naturally
- Two ramps connecting adjacent floors, modeled as continuous inclined surfaces
- Interior walls and obstacles creating a realistic floor layout
- Five cost terms: goal attraction, wall penalty, height adherence, smoothness, and agent repulsion
- Offscreen 3D rendering via `vedo` with frame-by-frame video export

## Table of Contents

1. [Installation & Setup](#1-installation--setup)
2. [Imports & Backend Configuration](#2-imports--backend-configuration)
3. [Simulation Parameters](#3-simulation-parameters)
4. [Building Geometry](#4-building-geometry)
   - 4.1 [Exits](#41-exits)
   - 4.2 [Ramps](#42-ramps)
   - 4.3 [Walls & Obstacles](#43-walls--obstacles)
5. [Geometry Helpers](#5-geometry-helpers)
6. [Ramp & Surface Height Functions](#6-ramp--surface-height-functions)
7. [Cost Function](#7-cost-function)
   - 7.1 [Goal Attraction (Soft-Min)](#71-goal-attraction--soft-min)
   - 7.2 [Wall Penalty](#72-wall-penalty)
   - 7.3 [Height-Adherence Cost](#73-height-adherence-cost)
   - 7.4 [Smoothness Term](#74-smoothness-term)
   - 7.5 [Agent Repulsion](#75-agent-repulsion)
   - 7.6 [Total Cost](#76-total-cost)
8. [Gradient Computation](#8-gradient-computation)
9. [Agent Initialization](#9-agent-initialization)
10. [3D Building Visualization](#10-3d-building-visualization)
11. [Simulation Loop with Real-Time Animation](#11-simulation-loop-with-real-time-animation)
12. [Summary & Discussion](#12-summary--discussion)

## 1. Installation & Setup

Install all required packages before running any other cells.

In [1]:
import sys
!{sys.executable} -m pip install numpy vedo
!{sys.executable} -m pip install -q vedo matplotlib
!{sys.executable} -m pip install -q vedo trame trame-vtk trame-vuetify
!{sys.executable} -m pip install imageio[ffmpeg]


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports & Backend Configuration

`vedo` is configured with offscreen rendering so the simulation runs in both VS Code notebooks and Colab without needing a GUI window or trame backend.

In [2]:
import numpy as np
from vedo import *

# Offscreen rendering - works everywhere (VS Code, Colab, no popup needed)
settings.default_backend = "vtk"


## 3. Simulation Parameters

All tunable constants are defined here. Changing these affects both the physics behavior and the visualization.

| Parameter | Symbol | Value | Description |
|-----------|--------|-------|-------------|
| Floor height | $H$ | 10 | Vertical separation between floors |
| Agents | $N$ | 20 | Number of simulated particles |
| Step size | $\alpha$ | 0.02 | Gradient descent learning rate |
| Soft-min temp | $\tau$ | 1.0 | Exit selection sharpness |
| Height stiffness | $w_h$ | 10 | Surface adherence weight |
| Smoothness weight | $w_s$ | 0.5 | Trajectory regularization weight |
| Repulsion radius | — | 1.0 | Personal space radius |
| Repulsion strength | — | 3.0 | Agent-agent repulsion weight |
| Max step | — | 0.5 | Step-size cap per iteration |
| Exit tolerance | — | 0.8 | Distance to count as evacuated |
| Steps | — | 600 | Total simulation iterations |

In [3]:
# ── Simulation parameters ──────────────────────────────────────────────────
H                 = 10       # floor height
num_agents        = 25      # number of agents
alpha             = 0.02     # gradient descent step size
tau               = 1.0      # soft-min temperature
wh                = 30.0     # height-adherence weight
ws                = 0.5      # smoothness weight
repulsion_radius  = 1.0      # personal space radius
repulsion_strength= 3.0      # repulsion penalty weight
max_step          = 0.5      # step-size cap
exit_tol          = 0.8      # distance to consider agent "evacuated"
exit_radius       = 1.0      # radius of the exit area
steps             = 600      # total simulation iterations

## 4. Building Geometry

### 4.1 Exits

Two exits are placed at opposite corners of the ground floor ($z = 0$). Placing them at opposite corners forces agents from different starting positions to travel in different directions, which makes the natural exit-splitting behavior of the soft-min more visible. No agent is ever assigned to a specific exit — the cost function handles selection entirely.

In [4]:
# ── Exits (ground floor only) ──────────────────────────────────────────────
exit1 = np.array([2.0,  2.0,  0.0])
exit2 = np.array([18.0, 18.0, 0.0])
exits = [exit1, exit2]

### 4.2 Ramps

Each ramp is defined by a centerline segment $[A, B]$ in the floor plan, start and end heights $z_0$ and $z_1$, and a width $r$ that forms the capsule footprint. Two ramps connect adjacent floors:

- **Ramp 1**: Floor 1 ($z = H$) → Ground floor ($z = 0$), running north–south through the building center
- **Ramp 2**: Floor 2 ($z = 2H$) → Floor 1 ($z = H$), running east–west through the center

Agents within distance $r$ of the centerline are on the ramp and follow the interpolated height function.

In [5]:
# ── Ramps ──────────────────────────────────────────────────────────────────
ramps = [
    {"A": np.array([10.0, 4.0]),  "B": np.array([10.0, 16.0]), "z0": H,   "z1": 0,   "width": 3.0},
    {"A": np.array([4.0,  10.0]), "B": np.array([16.0, 10.0]), "z0": 2*H, "z1": H,   "width": 3.0},
]


### 4.3 Walls & Obstacles

Walls are 2D line segments. The outer boundary is a $20 \times 20$ square. Interior walls add layout complexity per floor. This is a direct extension of the previous 2D floor-plan assignment — the same wall penalty is now applied inside a 3D building.

The ground floor includes a central box obstacle (four wall segments forming a closed rectangle) that forces agents to navigate around it rather than taking a straight path to the exit. This makes the emergent routing behavior more visible in the animation.

In [6]:
# ── Walls (2D line segments) ───────────────────────────────────────────────
# Outer boundary
walls = [
    ((0,0),  (20,0)),
    ((20,0), (20,20)),
    ((20,20),(0,20)),
    ((0,20), (0,0)),
    # Interior walls — ground floor corridor
    ((5,0),  (5,8)),
    ((5,12), (5,20)),
    ((15,0), (15,8)),
    ((15,12),(15,20)),
]

## 5. Geometry Helpers

### 5.1 Point-to-Segment Distance

Used by both the wall cost and ramp detection. For point $P$ and segment $[A, B]$:

$$t = \text{clip}\!\left(\frac{(P-A)\cdot(B-A)}{\|B-A\|^2},\; 0,\; 1\right), \qquad d(P,[A,B]) = \|P - (A + t(B-A))\|$$

In [7]:
def point_segment_distance(P, A, B):
    """Euclidean distance from point P to segment [A, B]."""
    v = B - A
    t = np.dot(P - A, v) / np.dot(v, v)
    t = np.clip(t, 0, 1)
    return np.linalg.norm(P - (A + t * v))

## 6. Ramp & Surface Height Functions

### 6.1 Ramp Height

For a ramp with centerline $[A, B]$ and heights $z_0$ (top) and $z_1$ (bottom), the height at any $(x, y)$ inside the footprint is:

$$z_{\text{ramp}}(x,y) = z_0 + (z_1 - z_0)\, u(x,y), \qquad u = \text{clip}\!\left(\frac{(P-A)\cdot(B-A)}{\|B-A\|^2}, 0, 1\right)$$

### 6.2 Surface Height

Outside any ramp the surface is a flat floor. The full piecewise surface blends between them:

$$z_{\text{surf}}(x,y) = \begin{cases} z_{\text{ramp}}(x,y) & \text{if } (x,y) \in \mathcal{R} \\ \lfloor z / H \rfloor \cdot H & \text{otherwise} \end{cases}$$

Floor-snapping uses integer division (truncate down) so agents always snap to the floor they are standing on, not the ceiling above.

In [8]:
def ramp_height(x, y, ramp):
    """Linear height interpolation along ramp centerline."""
    A, B = ramp["A"], ramp["B"]
    P = np.array([x, y])
    v = B - A
    u = np.clip(np.dot(P - A, v) / np.dot(v, v), 0, 1)
    return ramp["z0"] + (ramp["z1"] - ramp["z0"]) * u


def surface_height(x, y, z):

    P = np.array([x, y])

    # 1. Check ramps first
    for ramp in ramps:
        dist = point_segment_distance(P, ramp["A"], ramp["B"])
        if dist <= ramp["width"]:
            return ramp_height(x, y, ramp)

    # 2. Determine closest floor
    floors = [0.0, float(H), float(2*H)]

    target_z = min(floors, key=lambda f: abs(f - z))

    return target_z

## 7. Cost Function

The total cost driving every agent is:

$$C(p) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smooth}} + C_{\text{repulsion}}$$

No term hard-codes behavior — routing, floor transitions, and exit selection all emerge from the gradient of this combined landscape.

### 7.1 Goal Attraction — Soft-Min

A hard minimum over exits is non-differentiable, so the soft-min formulation is required:

$$C_{\text{goal}}(p) = -\tau \log \sum_{i=1}^{2} \exp\!\left(-\frac{\|p - p_i^{\text{exit}}\|^2}{\tau}\right)$$

Its gradient is a weighted combination of individual exit attractions, where each weight reflects how close and attractive that exit is:

$$\nabla C_{\text{goal}} = \sum_i w_i \nabla C_i, \qquad w_i = \frac{\exp(-C_i/\tau)}{\sum_j \exp(-C_j/\tau)}$$

Temperature $\tau = 1.0$ gives a smooth blend. Lower $\tau$ makes agents commit more sharply to one exit.

In [9]:
def goal_cost(p):

    costs = []

    for e in exits:
        d = np.linalg.norm(p - e)
        costs.append(d*d)

    costs = np.array(costs)

    return -tau * np.log(np.sum(np.exp(-costs / tau)))


### 7.2 Wall Penalty

Quadratic band penalty — activates when an agent comes within radius $r = 1.0$ of any wall segment:

$$C_{\text{walls}} = \sum_k \max(0,\; r - d_k(p))^2$$

This is the same formulation used in the previous 2D assignment, now applied in 3D using the XY projection of the agent's position.

In [10]:
def wall_cost(p):
    """
    Quadratic band penalty: C_walls = sum_k max(0, r - d_k)^2
    """
    cost = 0.0
    r = 1.0
    for w in walls:
        A = np.array(w[0], dtype=float)
        B = np.array(w[1], dtype=float)
        d = point_segment_distance(p[:2], A, B)
        if d < r:
            cost += (r - d)**2
    return cost

### 7.3 Height-Adherence (Surface) Cost

Keeps agents constrained to the building surfaces — floor slabs and ramp inclines:

$$C_{\text{height}} = w_h\,(z - z_{\text{surf}}(x,y))^2$$

When $w_h$ is too small, agents drift between floors. When too large, gradient steps shrink excessively. $w_h = 10$ was found to be a stable value experimentally.

In [11]:
def height_cost(p):

    x, y, z = p

    zsurf = surface_height(x, y, z)

    return wh * (z - zsurf)**2


### 7.4 Smoothness (Temporal Regularization)

Penalises abrupt jumps between consecutive positions:

$$C_{\text{smooth}} = w_s\,\|p_{k+1} - p_k\|^2$$

This reduces oscillation near walls and ramp edges where gradients can be large.

In [12]:
def smoothness_cost(p, p_prev):
    """
    Trajectory smoothness: C_smooth = ws * ||p - p_prev||^2
    """
    return ws * np.linalg.norm(p - p_prev)**2

### 7.5 Agent Repulsion

Short-range repulsion prevents agents from occupying the same space:

$$C_{\text{repulsion}} = \sum_{j \neq i} \varphi(\|p_i - p_j\|), \qquad \varphi(d) = s \cdot \max(0,\, r_{\text{rep}} - d)^2$$

Same quadratic band family as the wall penalty, so gradient magnitude scales smoothly with overlap depth.

In [13]:
def repulsion_cost(p, agents):
    """
    Short-range quadratic repulsion from all other agents.
    """
    cost = 0.0
    for a in agents:
        d = np.linalg.norm(p - a)
        if 0 < d < repulsion_radius:
            cost += repulsion_strength * (repulsion_radius - d)**2
    return cost

### 7.6 Total Cost

$$C(p) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smooth}} + C_{\text{repulsion}}$$

In [14]:
def total_cost(p, p_prev, agents):
    return (
        goal_cost(p)
        + wall_cost(p)
        + height_cost(p)
        + smoothness_cost(p, p_prev)
        + repulsion_cost(p, agents)
    )

## 8. Gradient Computation

Gradients are computed numerically using central finite differences:

$$\frac{\partial C}{\partial p_i} \approx \frac{C(p + \epsilon\, e_i) - C(p - \epsilon\, e_i)}{2\epsilon}, \qquad \epsilon = 10^{-3}$$

This avoids having to derive analytical gradients for each cost term, which is especially convenient for the piecewise surface height function.

In [15]:
def gradient(p, p_prev, agents, eps=1e-3):
    """Finite-difference gradient of total_cost w.r.t. p."""
    g = np.zeros(3)
    for i in range(3):
        dp = np.zeros(3)
        dp[i] = eps
        g[i] = (total_cost(p + dp, p_prev, agents)
               - total_cost(p - dp, p_prev, agents)) / (2 * eps)
    return g

## 9. Agent Initialization

Agents are seeded with `seed=42` for reproducibility. Each gets a random $(x,y)$ within the floor bounds and is placed on one of the three floors uniformly at random.

In [16]:
np.random.seed(42)

agents = []
for _ in range(num_agents):
    x     = np.random.uniform(2, 18)
    y     = np.random.uniform(2, 18)
    floor = np.random.choice([0, H, 2*H])
    agents.append(np.array([x, y, float(floor)]))

agents      = np.array(agents)
prev_agents = agents.copy()          # needed for smoothness term
active      = np.ones(num_agents, dtype=bool)  # False when agent exits

print(f"Initialized {num_agents} agents across floors 0, {H}, {2*H}")

Initialized 25 agents across floors 0, 10, 20


## 10. 3D Building Visualization

#@title Building Scene Construction

The 3D scene follows the stacked cutaway isometric style from the assignment figure:

- **Floor slabs**: thick concrete-coloured boxes, one per floor, separated by a visible gap
- **Outer walls**: south and west faces are full height; north and east are cut short so the interior is visible from the camera
- **Interior walls and obstacles**: short dividers per floor plus the box obstacle on the ground floor
- **Ramps**: inclined quads with solid side cheeks dropping to the lower floor level
- **Exit markers**: green door frames with a cone arrow and glow sphere above
- **Agents**: red spheres that turn grey on evacuation

In [17]:

import numpy as np
from vedo import *

# ═══════════════════════════════════════════════════════════════════
# DETAILED 3-FLOOR BUILDING VISUALIZATION
# Inspired by the stacked cutaway isometric style in the assignment
# ═══════════════════════════════════════════════════════════════════

W   = 20.0   # building width/depth
T   = 0.25   # wall thickness
FH  = H - 1  # usable floor-to-ceiling height (leave 1 unit gap between floors)

all_objects = []

# ───────────────────────────────────────────────────────────────────
# Helper: flat panel mesh from corner points
# ───────────────────────────────────────────────────────────────────
def panel(x0,y0,z0, x1,y1,z1, x2,y2,z2, x3,y3,z3, color="white", alpha=0.8):
    pts   = [[x0,y0,z0],[x1,y1,z1],[x2,y2,z2],[x3,y3,z3]]
    faces = [[0,1,2,3]]
    return Mesh([pts,faces]).color(color).alpha(alpha).lw(0)

def box_mesh(cx,cy,cz, sx,sy,sz, color="white", alpha=0.85):
    return Box(pos=(cx,cy,cz), size=(sx,sy,sz)).color(color).alpha(alpha)

# ───────────────────────────────────────────────────────────────────
# FLOOR SLABS  (thick, with a warm concrete colour)
# ───────────────────────────────────────────────────────────────────
SLAB = 0.4
for k, col in enumerate(["#d4c5a9","#cfc0a3","#c8b89a"]):
    z = k * H
    all_objects.append(box_mesh(W/2, W/2, z - SLAB/2, W+T*2, W+T*2, SLAB, col, 0.95))

# ───────────────────────────────────────────────────────────────────
# OUTER WALLS  — only South & West are full-height (visible faces)
#               North & East are cut low so interior is visible
# ───────────────────────────────────────────────────────────────────
def outer_walls_for_floor(z_base, full_h, cut_h, wall_col="#f0ece4", alpha=0.82):
    walls = []
    # South wall  (y = 0) — full height
    walls.append(box_mesh(W/2, 0,    z_base+full_h/2, W+T*2, T, full_h, wall_col, alpha))
    # West wall   (x = 0) — full height
    walls.append(box_mesh(0,   W/2,  z_base+full_h/2, T, W,   full_h, wall_col, alpha))
    # North wall  (y = W) — cut short so we can see inside
    walls.append(box_mesh(W/2, W,    z_base+cut_h/2,  W+T*2, T, cut_h, wall_col, alpha*0.7))
    # East wall   (x = W) — cut short
    walls.append(box_mesh(W,   W/2,  z_base+cut_h/2,  T, W,   cut_h, wall_col, alpha*0.7))
    return walls

for k in range(3):
    z = k * H
    all_objects += outer_walls_for_floor(z, FH, FH*0.35)

# ───────────────────────────────────────────────────────────────────
# INTERIOR WALLS  — short dividers that add layout complexity
# Same layout on every floor (slightly different per floor)
# ───────────────────────────────────────────────────────────────────
def interior_wall(x0,y0,x1,y1, z_base, height, col="#e8dfd0", alpha=0.80):
    A = np.array([x0,y0],dtype=float)
    B = np.array([x1,y1],dtype=float)
    mid = (A+B)/2
    L   = np.linalg.norm(B-A)
    ang = np.degrees(np.arctan2(y1-y0, x1-x0))
    b   = Box(pos=(mid[0], mid[1], z_base+height/2), size=(L, T, height))
    b.rotate_z(ang)
    return b.color(col).alpha(alpha)

# Ground floor
z=0; h=FH*0.6
all_objects += [
    interior_wall(5,  0,  5,  8,  z, h),
    interior_wall(5,  12, 5,  W,  z, h),
    interior_wall(15, 0,  15, 8,  z, h),
    interior_wall(15, 12, 15, W,  z, h),
    interior_wall(5,  8,  9,  8,  z, h*0.7),
    interior_wall(11, 8,  15, 8,  z, h*0.7),
    interior_wall(8,  12, 12, 12, z, h*0.7),
]

# First floor
z=H; h=FH*0.55
all_objects += [
    interior_wall(7,  0,  7,  9,  z, h),
    interior_wall(7,  13, 7,  W,  z, h),
    interior_wall(13, 0,  13, 9,  z, h),
    interior_wall(13, 13, 13, W,  z, h),
    interior_wall(7,  9,  13, 9,  z, h*0.7),
]

# Second floor
z=2*H; h=FH*0.5
all_objects += [
    interior_wall(6,  0,  6,  10, z, h),
    interior_wall(6,  14, 6,  W,  z, h),
    interior_wall(14, 5,  14, W,  z, h),
    interior_wall(6,  10, 10, 10, z, h*0.65),
    interior_wall(10, 14, 10, W,  z, h*0.65),
]

# ───────────────────────────────────────────────────────────────────
# RAMP SURFACES  — properly oriented inclined quads
# ───────────────────────────────────────────────────────────────────
def make_ramp_mesh(ramp, col="#8B6914"):
    A2 = ramp["A"]; B2 = ramp["B"]
    z0 = ramp["z0"]; z1 = ramp["z1"]; w = ramp["width"]
    dx = B2[0]-A2[0]; dy = B2[1]-A2[1]
    L  = np.sqrt(dx*dx+dy*dy)
    px, py = -dy/L*w, dx/L*w
    pts = [
        [A2[0]+px, A2[1]+py, z0],
        [A2[0]-px, A2[1]-py, z0],
        [B2[0]-px, B2[1]-py, z1],
        [B2[0]+px, B2[1]+py, z1],
    ]
    faces = [[0,1,2,3]]
    m = Mesh([pts,faces]).color(col).alpha(0.92).lw(1)
    # Side panels to make the ramp look like a solid ramp
    side1 = Mesh([[ pts[0], pts[3],
                    [pts[3][0],pts[3][1],z1-H],
                    [pts[0][0],pts[0][1],z0-H]], [[0,1,2,3]]]).color("#6b5210").alpha(0.85)
    side2 = Mesh([[ pts[1], pts[2],
                    [pts[2][0],pts[2][1],z1-H],
                    [pts[1][0],pts[1][1],z0-H]], [[0,1,2,3]]]).color("#6b5210").alpha(0.85)
    return [m, side1, side2]

for r in ramps:
    all_objects += make_ramp_mesh(r)

# ───────────────────────────────────────────────────────────────────
# EXIT MARKERS  — green archway + glow sphere
# ───────────────────────────────────────────────────────────────────
for ex in exits:
    # Door opening (green rectangle on wall)
    all_objects.append(box_mesh(ex[0], ex[1], 1.2, 1.8, 0.15, 2.4, "#00ff88", 0.95))
    # Glow sphere above
    all_objects.append(Sphere(pos=(ex[0], ex[1], 2.8), r=0.55).color("#00ff44").alpha(0.9))

# ───────────────────────────────────────────────────────────────────
# AGENT SPHERES
# ───────────────────────────────────────────────────────────────────
agent_spheres = [Sphere(a, r=0.38).color("tomato").alpha(1.0) for a in agents]
all_objects  += agent_spheres

print("Detailed scene built:", len(all_objects), "objects")


Detailed scene built: 67 objects


## 11. Simulation Loop with Real-Time Animation

#@title Simulation & Video Export

Each step:

1. Compute the finite-difference gradient of the total cost for each active agent
2. Apply gradient descent: $p \leftarrow p - \alpha \nabla C$
3. Cap the step magnitude at `max_step = 0.5` to prevent numerical blowup
4. Deactivate any agent within `exit_tol` of an exit
5. Capture a screenshot every 2 steps

After the loop, `imageio` assembles all frames into an MP4 at 12 fps.

In [18]:
# ── Plotter (offscreen) ───────────────────────────────────────────────────
plt = Plotter(bg="#1a1a2e", bg2="#16213e", title="Building Evacuation",
              offscreen=True, size=(1280, 720))

plt.show(all_objects, interactive=False,
         camera=dict(
             pos=(60, -40, 52),
             focal_point=(10, 10, 12),
             viewup=(0, 0, 1)
         ))

# ── Simulation + frame capture ─────────────────────────────────────────────
import os, imageio
from IPython.display import Video, display

os.makedirs("frames", exist_ok=True)
frames_paths = []

for step in range(steps):
    if not np.any(active):
        print(f"All agents evacuated at step {step}!")
        break

    new_agents = agents.copy()

    for i, p in enumerate(agents):
        if not active[i]:
            continue

        for e in exits:
            # XY distance only — agent deactivates when directly above exit
            if np.linalg.norm(p[:2] - e[:2]) < exit_tol and p[2] < H * 0.5:
                active[i] = False
                agent_spheres[i].color("grey").alpha(0.25)
                break

        if not active[i]:
            continue

        others = agents[np.arange(num_agents) != i]
        g      = gradient(p, prev_agents[i], others)
        new_p  = p - alpha * g

        step_vec  = new_p - p
        step_size = np.linalg.norm(step_vec)
        if step_size > max_step:
            step_vec = step_vec / step_size * max_step
            new_p    = p + step_vec

        new_agents[i] = new_p
        agent_spheres[i].pos(new_p)

    prev_agents = agents.copy()
    agents      = new_agents

    if step % 3 == 0:
        path = f"frames/frame_{step:04d}.png"
        plt.screenshot(path)
        frames_paths.append(path)

    if step % 50 == 0:
        print(f"Step {step:3d} | Evacuated: {int(np.sum(~active))}/{num_agents}")

plt.close()

# ── Assemble video ─────────────────────────────────────────────────────────
video_path = "evacuation.mp4"
with imageio.get_writer(video_path, fps=20, codec="libx264",
                        ffmpeg_params=["-pix_fmt", "yuv420p"]) as writer:
    for fp in frames_paths:
        writer.append_data(imageio.imread(fp))

print("\nVideo saved:", video_path)
display(Video(video_path, embed=True, width=700))


Step   0 | Evacuated: 0/25
Step  50 | Evacuated: 3/25
Step 100 | Evacuated: 6/25
Step 150 | Evacuated: 6/25
Step 200 | Evacuated: 6/25
Step 250 | Evacuated: 6/25
Step 300 | Evacuated: 6/25
Step 350 | Evacuated: 6/25
Step 400 | Evacuated: 6/25
Step 450 | Evacuated: 6/25
Step 500 | Evacuated: 6/25
Step 550 | Evacuated: 6/25


C:\Users\cajus\AppData\Local\Temp\ipykernel_38768\1593027469.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(fp))



Video saved: evacuation.mp4


## 12. Summary & Discussion

### Cost Function Design

| Term | Formula | Role |
|------|---------|------|
| $C_{\text{goal}}$ (soft-min) | $-\tau \log \sum_i e^{-C_i/\tau}$ | Attracts agents to exits; enables natural group splitting |
| $C_{\text{walls}}$ (quadratic band) | $\sum_k \max(0, r-d_k)^2$ | Keeps agents inside corridors and away from obstacles |
| $C_{\text{height}}$ | $w_h(z - z_{\text{surf}})^2$ | Constrains agents to floors and ramps; enables floor transitions |
| $C_{\text{smooth}}$ | $w_s \|p_{k+1} - p_k\|^2$ | Stabilises trajectories; reduces oscillation near boundaries |
| $C_{\text{repulsion}}$ | $\sum_{j \neq i} s \max(0, r-d_{ij})^2$ | Models crowd interaction; prevents agent overlap |

### Design Decisions

**Soft-min temperature** ($\tau = 1.0$): balances between sharp exit preference ($\tau \to 0$) and uniform blending ($\tau \to \infty$). Agents near the center of the floor plan split between exits naturally rather than all converging on one.

**Step-size capping** (max = 0.5): prevents numerical instability from large gradient steps near walls or ramp edges, where the cost surface has steep curvature.

**Ramp model**: the capsule-footprint with linear height interpolation produces a continuously differentiable surface, which keeps gradient descent stable through floor transitions with no discrete floor-switching logic.

**Obstacles**: the box obstacle on the ground floor is a direct extension of the 2D floor-plan assignment. Agents cannot cut straight to the exit and must navigate around it, which makes the emergent routing behavior clearly visible in the animation.

### Observations

Agents on upper floors move toward the ramp entrance before descending — this routing is entirely emergent from the height-adherence cost. No rule directs them to the ramp; the cost landscape simply makes standing on the ramp cheaper than hovering in mid-air between floors.

The two exits attract roughly equal numbers of agents when agents start uniformly distributed, with natural clustering near each exit depending on starting position. The smoothness term visibly reduces jitter near wall boundaries.

Was unable to achieve the intended results. This could be due to agents goal cost function having the goal as closer tha the ramp due the goal being closer even if it ona different Y level.